# Exp05 · dua target pada tingkat episode

Jalankan semua sel pada runtime CPU. V3 memakai Train dan validation resmi. V4 menahan seluruh episode satu peserta bergantian. Kedua hasil masuk ke satu ZIP. Kedua subfolder paket harus berada langsung di Dataset.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
DATASET_ROOT = PROJECT_ROOT / 'Dataset'
OUTPUT_ROOT = PROJECT_ROOT / 'models/batch-06/exp05/outputs'

## Validasi paket, fitur sensor dan metrik

In [ ]:
import hashlib
import json
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import RobustScaler

HRV = ["rmssd_ms", "sdnn_ms"]
EYE = ["blink_rate_per_minute", "ear_left_mean", "ear_left_std", "ear_right_mean",
       "ear_right_std", "iris_size_estimate_mean", "iris_size_estimate_std", "iris_size_delta_mean"]
CORE = HRV + EYE
ID = ["participant_id", "episode_id"]
SEED = 42


def read_bundle(root, case):
    proxy = case == "v3"
    package = root / ("v3_behavioral_proxy" if proxy else "v4_self_reported_rating")
    target = "cognitive_load_label" if proxy else "difficulty_rating"
    labels = ["Low", "Medium", "High"] if proxy else list(range(1, 11))
    window_dir = Path("window_15s_step_5s")
    hashes = {}

    def read(relative, columns):
        path = package / relative
        hashes[str(Path(relative).as_posix())] = hashlib.sha256(path.read_bytes()).hexdigest()
        return pd.read_csv(path, usecols=columns)

    def attach(left, right, keys, validation):
        merged = left.merge(right, on=keys, how="left", validate=validation, indicator=True)
        if not merged.pop("_merge").eq("both").all():
            raise ValueError(f"CSV pendamping tidak cocok pada {keys}.")
        return merged

    keys = ["window_id", "participant_id", "session_id", target]
    data = read(window_dir / "good_acceptable.csv", keys + CORE)
    assignments = read(window_dir / "window_assignments.csv",
                       keys + ["episode_id"] + (["split"] if proxy else []))
    for table in (data, assignments):
        if table.window_id.duplicated().any() or table[keys].isna().any().any():
            raise ValueError("Kunci window tidak lengkap atau window_id berulang.")
    data = attach(data, assignments, keys, "one_to_one")
    evidence = read("episode_labels.csv" if proxy else "episode_ratings.csv",
                    ID + ["session_id", target])
    if evidence.episode_id.isna().any() or evidence.episode_id.duplicated().any():
        raise ValueError("episode_id pada bukti harus unik dan lengkap.")
    data = attach(data, evidence, ID + ["session_id", target], "many_to_one")
    if set(data[target]) != set(labels) or not np.isfinite(data[CORE].to_numpy(float)).all():
        raise ValueError("Periksa kelas target dan kelengkapan sepuluh fitur core.")
    if proxy:
        splits = read("participant_splits.csv", ["participant_id", "split"])
        if splits.participant_id.duplicated().any() or set(splits.split) != {"Train", "validation", "Test"}:
            raise ValueError("Periksa pembagian peserta v3.")
        data = attach(data, splits, ["participant_id", "split"], "many_to_one")
        weights = read(window_dir / "train_sample_weights.csv", ["window_id", *ID, target, "sample_weight"])
        train = data.split.eq("Train")
        if set(weights.window_id) != set(data.loc[train, "window_id"]) or weights.window_id.duplicated().any():
            raise ValueError("Daftar bobot berbeda dari window Train.")
        checked = attach(data.loc[train], weights, ["window_id", *ID, target], "one_to_one")
        if not np.isfinite(checked.sample_weight).all() or not checked.sample_weight.gt(0).all():
            raise ValueError("Bobot Train harus positif dan finite.")
        data["sample_weight"] = data.window_id.map(checked.set_index("window_id").sample_weight)
    return data.rename(columns={target: "actual"}), labels, hashes

def scores(table, labels):
    actual, predicted = table.actual, table.predicted
    row = dict(n=len(table), participants=table.participant_id.nunique(),
               accuracy=accuracy_score(actual, predicted),
               macro_f1=f1_score(actual, predicted, labels=labels, average="macro", zero_division=0))
    if len(labels) == 10:
        distance = (actual - predicted).abs()
        row.update(mae=distance.mean(), within_one=distance.le(1).mean())
    return row

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")

## Prediksi dari window pertama dan median seluruh episode

In [ ]:
"""Episode-level comparison for the paired Colab notebook. Shared CSV loader is embedded at build time."""

from sklearn.linear_model import QuantileRegressor
from sklearn.preprocessing import StandardScaler


def episode_views(root, case, windows):
    package = "v3_behavioral_proxy" if case == "v3" else "v4_self_reported_rating"
    source = root / package / "window_15s_step_5s/good_acceptable.csv"
    times = pd.read_csv(source, usecols=["window_id", "window_start"])
    times["window_start"] = pd.to_datetime(times.window_start, utc=True, format="mixed", errors="raise")
    data = windows.merge(times, on="window_id", how="left", validate="one_to_one")
    if data.window_start.isna().any() or data.duplicated(["episode_id", "window_start"]).any():
        raise ValueError("Awal window kosong atau ambigu dalam episode.")
    if case == "v3":
        data = data[data.split.isin(["Train", "validation"])]
    ordered = data.sort_values(["episode_id", "window_start"])
    meta = ordered.drop_duplicates("episode_id").set_index("episode_id")
    keys = ["participant_id", "actual"] + (["split"] if case == "v3" else [])
    episodes = meta[keys].copy()
    groups = ordered.groupby("episode_id")
    if groups[keys].nunique().gt(1).any().any():
        raise ValueError("Metadata episode tidak konsisten.")
    if case == "v3":
        # Published weights allocate mass hierarchically down to windows.
        episodes["weight"] = groups.sample_weight.sum(min_count=1)
    return episodes, {"first": meta[CORE], "median": groups[CORE].median().loc[meta.index]}


def evaluate_fold(case, episodes, views, fit, held, labels, fold):
    train, test = episodes.loc[fit], episodes.loc[held]
    if set(train.participant_id) & set(test.participant_id):
        raise ValueError("Peserta evaluasi muncul pada fitting.")
    y = train.actual
    weights = train.weight / train.weight.mean() if case == "v3" else np.ones(len(train))
    predictions = []

    def record(name, predicted, estimate=None):
        rows = test[["participant_id", "actual"]].reset_index()
        rows["case"], rows["fold"], rows["model"] = case, fold, name
        rows["predicted"] = predicted
        if estimate is not None:
            rows["continuous_rating"] = estimate
        predictions.append(rows)

    counts = y.value_counts().reindex(labels, fill_value=0)
    record("dummy_mode", counts.idxmax())
    if case == "v4":
        record("dummy_median", y.quantile(0.5, interpolation="lower"))
    for view, x in views.items():
        scaler = StandardScaler().fit(x.loc[fit], sample_weight=weights)
        x_train = scaler.transform(x.loc[fit])
        x_held = scaler.transform(x.loc[held])
        classifier = LogisticRegression(C=0.1, max_iter=3000)
        classifier.fit(x_train, y, sample_weight=weights)
        record("logistic_" + view, classifier.predict(x_held))
        if case == "v4":
            reg = QuantileRegressor(quantile=0.5, alpha=0.10, solver="highs")
            reg.fit(x_train, y)
            estimate = reg.predict(x_held)
            rating = np.floor(np.clip(estimate, 1, 10) + 0.5).astype(int)
            record("quantile_" + view, rating, estimate)
    return predictions


def run_episodes(dataset_root, output_root):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out = output_root / f"episodes_{stamp}_{uuid4().hex[:8]}"
    out.mkdir(parents=True, exist_ok=False)
    manifest = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                "sklearn": sklearn.__version__, "inputs": {}, "features": CORE,
                "views": ["first eligible window", "coordinatewise median of all eligible windows"],
                "logistic_C": 0.1, "quantile": 0.5, "quantile_alpha": 0.1,
                "primary": {"v3": "logistic_median", "v4": "quantile_median"}}
    all_metrics, all_people, all_confusions = [], [], []
    for case in ("v3", "v4"):
        windows, labels, manifest["inputs"][case] = read_bundle(dataset_root, case)
        episodes, views = episode_views(dataset_root, case, windows)
        if case == "v3":
            folds = [("validation", episodes.split.eq("Train"), episodes.split.eq("validation"))]
        else:
            folds = [(f"LOPO_{i}", episodes.participant_id.ne(person), episodes.participant_id.eq(person))
                     for i, person in enumerate(sorted(episodes.participant_id.unique()), 1)]
        predictions = []
        memberships = []
        for fold, fit, held in folds:
            predictions.extend(evaluate_fold(case, episodes, views, fit, held, labels, fold))
            for role, mask in [("fit", fit), ("held", held)]:
                membership = episodes.loc[mask, ["participant_id", "actual"]].reset_index()
                memberships.append(membership.assign(fold=fold, role=role))
        pred = pd.concat(predictions, ignore_index=True)
        expected = episodes.index[folds[0][2]] if case == "v3" else episodes.index
        for model, subset in pred.groupby("model"):
            if subset.episode_id.duplicated().any() or set(subset.episode_id) != set(expected):
                raise ValueError("Prediksi episode ganda atau tidak lengkap.")
            info = {"case": case, "model": model}
            all_metrics.append({**info, **scores(subset, labels)})
            for person, group in subset.groupby("participant_id"):
                all_people.append({**info, "participant_id": person, **scores(group, labels)})
            matrix = confusion_matrix(subset.actual, subset.predicted, labels=labels)
            for label, counts in zip(labels, matrix):
                all_confusions.append({**info, "actual": label, "support": counts.sum(),
                                       **dict(zip([f"pred_{k}" for k in labels], counts))})
        pred.to_csv(out / f"{case}_episode_predictions.csv", index=False)
        pd.concat(memberships).to_csv(out / f"{case}_folds.csv", index=False)
    save_json(out / "manifest.json", manifest)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(out / "metrics.csv", index=False)
    pd.DataFrame(all_people).to_csv(out / "participant_metrics.csv", index=False)
    pd.DataFrame(all_confusions).to_csv(out / "confusion.csv", index=False)
    archive = shutil.make_archive(str(out), "zip", root_dir=out.parent, base_dir=out.name)
    print(metrics.to_string(index=False))
    print("ZIP untuk dikembalikan:", archive)
    return Path(archive)

## Jalankan kedua kasus dan simpan ZIP

In [ ]:
archive = run_episodes(DATASET_ROOT, OUTPUT_ROOT)